# Suite LST — Lists

A property whose schema is an `OfIndexed` holds a list: ordered items, each a value of one item schema, read as a
tuple and written over the wire as an array. The value objects in a list belong to the list's owner: they have
identities, may have adjacencies, are copied and removed with their owner, and are linked by `$id`.

In [ ]:
from mbse.Schemas.Framework import Comparison as C, JSON, Plain, Proxies, Reachable, Schemas as S, Validators, YAML
from mbse.Schemas.Framework.Errors import DecodeError
from support import raises, text

store = Proxies.OfStore()


def list_of(spec):
    """An `OfAny` Spec for a list of `spec`."""
    return lambda t: t.as_indexed(lambda i: i.of(spec))


def keyed(key, spec):
    """An `OfAny` Spec for a list of `spec` keyed by `key`."""
    return lambda t: t.as_indexed(lambda i: i.key(key).of(spec))


Str = lambda t: t.as_native(str)
Int = lambda t: t.as_native(int)
Port = S.OfObject.Builder().properties(text("name")).create()
Wire = S.OfRelation.Builder().links("source", "sink").create()
Float = lambda t: t.as_native(float)
Cell = S.OfObject.Builder().properties(text("r", int), text("c", int), text("w", float),
                                       lambda p: p.name("path").of(list_of(Int)),
                                       lambda p: p.name("meta").of(keyed(Str, Str))).create()
Rack = S.OfObject.Builder().properties(text("name"), lambda p: p.name("port").of(Port)).create()
Shape = S.OfUnion.Builder().branches(lambda b: b.name("circle").of(lambda t: t.as_native(float)),
                                     lambda b: b.name("label").of(Str)).create()
Part = S.OfObject.Builder().name("Part").ref().properties(
    text("name"), lambda p: p.name("tags").of(list_of(Str)), lambda p: p.name("ports").of(list_of(Port)),
    lambda p: p.name("grid").of(list_of(list_of(Int))), lambda p: p.name("banks").of(list_of(list_of(Port))),
    lambda p: p.name("shapes").of(list_of(Shape)), lambda p: p.name("racks").of(list_of(Rack)),
    lambda p: p.name("row").of(lambda t: t.as_indexed(lambda i: i.of(Int).extent(1, 3))),
    lambda p: p.name("attrs").of(keyed(Str, Float)), lambda p: p.name("weights").of(keyed(Float, Str)),
    lambda p: p.name("flags").of(keyed(lambda t: t.as_native(bool), Str)),
    lambda p: p.name("named").of(keyed(lambda t: t.as_native(bytes), Port)),
    lambda p: p.name("cells").of(keyed(Cell, Int))).create()
Hub = S.OfObject.Builder().name("Hub").ref().properties(text("name")).relations(
    lambda r: r.name("feeds").of(Wire).me("sink")).create()
S.OfObject.Builder(Port).relations(lambda r: r.name("wires").of(Wire).me("source"),
                                   lambda r: r.name("fed").of(Wire).me("sink")).update()
Labels = S.OfRelation.Builder().links("part", "hub").properties(lambda p: p.name("labels").of(list_of(Str)),
                                                              lambda p: p.name("notes").of(keyed(Str, Str))).create()
S.OfObject.Builder(Part).relations(lambda r: r.name("labelled").of(Labels).me("part")).update()
S.OfObject.Builder(Hub).relations(lambda r: r.name("labels").of(Labels).me("hub")).update()
store.register(Part)
store.register(Hub)
B = store


class Written:
    """A `Visitable` Part whose `accept` runs `write(visitor)`: data that proxies do not produce."""

    def __init__(self, write):
        self._write = write

    def identity(self):
        return id(self)

    def schema_name(self):
        return "Part"

    def owner(self):
        return None

    def accept(self, visitor):
        self._write(visitor)


def snapshot(**values):
    """A one-object Part snapshot with `values`."""
    return {"root": "s0", "objects": {"s0": values}}


def decode_error(plain, match):
    with raises(DecodeError, match=match):
        Plain.FromPlain(B)(Part, plain)

## LST-01 · `OfIndexed` holds one item schema of any kind, validated, and never a reference object schema

In [ ]:
assert Part.validate() == [] and Port.validate() == [] and Hub.validate() == [] and Labels.validate() == []
Tags = S.OfIndexed.Builder().of(Str).create()
assert isinstance(Tags, S.OfIndexed.Data) and isinstance(Tags.item, S.OfNative.Data) and Tags.validate() == []
assert S.OfAny.Builder().as_indexed(Tags).create() is Tags and S.OfIndexed.resolve(Tags) is Tags
assert isinstance(Part.properties["grid"].type.item, S.OfIndexed.Data)  # lists of lists
assert isinstance(Part.properties["shapes"].type.item, S.OfUnion.Data)
# The item's own problems are the list's.
assert S.OfIndexed.Data().validate() == ["item: not a schema: None"]
Wide = S.OfIndexed.Builder().of(lambda t: t.as_native(lambda n: n.type(int).bits(0))).create()
assert Wide.validate() == ["item: a width in bits must be a positive int, got 0"]
Holder = S.OfObject.Builder().properties(lambda p: p.name("wide").of(Wide)).create()
assert Holder.validate() == ["property 'wide': item: a width in bits must be a positive int, got 0"]
# A reference object schema is never an item's, at any depth, nor through a union.
Card = S.OfObject.Builder().ref().create()
Either = S.OfUnion.Builder().branches(lambda b: b.name("cards").of(list_of(Card)),
                                      lambda b: b.name("none").of(list_of(Int))).create()
Holder = S.OfObject.Builder().properties(lambda p: p.name("cards").of(list_of(Card)),
                                         lambda p: p.name("deep").of(list_of(list_of(Card))),
                                         lambda p: p.name("either").of(Either)).create()
assert Holder.validate() == ["property 'cards': a reference object schema cannot be a property's type",
                             "property 'deep': a reference object schema cannot be a property's type",
                             "property 'either': a reference object schema cannot be a property's type"]
# An entry's value objects have no adjacencies, in lists too.
Pair = S.OfRelation.Builder().links("a", "b").properties(lambda p: p.name("ports").of(list_of(Port))).create()
assert Pair.validate() == ["property 'ports': a value object held by an entry cannot have adjacencies"]
Pin = S.OfObject.Builder().properties(text("name")).create()
assert S.OfRelation.Builder().links("a", "b").properties(lambda p: p.name("pins").of(list_of(Pin))).create().validate() == []
# Builders finalize as every builder does.
copied = S.OfIndexed.Builder(Tags).clone()
assert copied is not Tags and copied.item is Tags.item
assert Tags == Tags and copied != Tags  # schema data are equal only to themselves
assert S.OfIndexed.Builder(Tags).of(Int).update() is Tags and Tags.item.type is int
with raises(ValueError, match="create() is only valid without a source instance"):
    S.OfIndexed.Builder(Tags).create()

## LST-02 · Proxies read a list as a read-only sequence and set it from a sequence of items

In [ ]:
p = B.Part().name("p").tags(["a", "b"]).grid(((1, 2), [3])).shapes(
    [lambda u: u.circle(1.5), lambda u: u.label("x")]).ports([lambda o: o.name("in")]).create()
assert p.tags == ("a", "b") and isinstance(p.tags, tuple) and p.grid == ((1, 2), (3,))
assert p.shapes[0].circle == 1.5 and p.shapes[1].label == "x" and p.ports[0].name == "in"
assert p.ports[0].owner() is p and p.shapes[1].owner() is p
assert B.member(p, "tags") == ("a", "b")
p = B.Part(p).tags(("c",)).update()
assert p.tags == ("c",)
p = B.Part(p).tags([]).update()
assert p.tags == ()  # an empty list is a value, unlike an absent one
p = B.Part(p).clear("tags").update()
with raises(AttributeError, match="property 'tags' is not set"):
    p.tags
with raises(AttributeError, match="read-only"):
    p.tags = ()
# An item written with no value is left out: a union value without a branch is no value.
assert len(B.Part().shapes([lambda u: u, lambda u: u.label("y")]).create().shapes) == 1
# Items are written under the item schema, and a list takes a list or a Spec.
with raises(TypeError, match="property 'tags' takes a list or a Spec, got str"):
    B.Part().tags("ab")
with raises(TypeError, match="property 'ports' takes a value object or a Spec, got str"):
    B.Part().ports(["x"])
with raises(TypeError, match="property 'name' does not hold a list"):
    B.Part().property("name", lambda p: p.value(lambda a: a.as_indexed(lambda items: None)))
with raises(TypeError, match="property 'tags' does not hold an object"):
    B.Part().property("tags", lambda p: p.value(lambda a: a.as_object(lambda o: None)))

## LST-03 · A Spec edits a list through its builder, keeping the identities of its value objects

In [ ]:
p = B.Part().ports([lambda o: o.name("a"), lambda o: o.name("b")]).create()
a, b = p.ports
p = B.Part(p).ports(lambda items: items.item(0, lambda v: v.as_object(lambda o: o.name("a2"))).append(
    lambda v: v.as_object(lambda o: o.name("c")))).update()
assert p.ports[0] is a and a.name == "a2" and p.ports[1] is b and p.ports[2].name == "c" and p.ports[2].owner() is p
names = []
B.Part(p).ports(lambda items: items.items(lambda v: v.as_object(lambda o: names.append(o.has("name")))))
assert names == [True, True, True]
for bad in (3, -1):
    with raises(LookupError, match=f"the list has no item {bad}"):
        B.Part(p).ports(lambda items: items.item(bad, lambda v: None))
    with raises(LookupError, match=f"the list has no item {bad}"):
        B.Part(p).ports(lambda items: items.remove(bad))
# Value objects in a list link each other, and to reference objects.
hub = B.Hub().name("hub").create()
p = B.Part(p).ports(lambda items: items.item(0, lambda v: v.as_object(lambda o: o.wires(lambda x: x.sink(b)).wires(
    lambda x: x.sink(hub))))).update()
assert p.ports[0] is a
assert Plain.ToPlain(store)(Part, p)["objects"]["s0"]["ports"][:2] == [
    {"$id": "s1", "name": "a2", "wires": [{"sink": {"$ref": "s2"}}, {"sink": {"$ref": "s3", "$schema": "Hub"}}]},
    {"$id": "s2", "name": "b", "fed": [{"source": {"$ref": "s1"}}]}]
# Removing an item removes it and the entries linking it; the others keep their identities.
p = B.Part(p).ports(lambda items: items.remove(1)).update()
assert p.ports == (a, p.ports[1]) and [x.name for x in p.ports] == ["a2", "c"]
assert Plain.ToPlain(store)(Part, p)["objects"]["s0"]["ports"][0]["wires"] == [{"sink": {"$ref": "s1", "$schema": "Hub"}}]
# Lists of lists keep identities too, by position; a list appended is new.
p = B.Part().banks([[lambda o: o.name("x")]]).create()
x = p.banks[0][0]
p = B.Part(p).banks(lambda rows: rows.item(0, lambda v: v.as_indexed(lambda row: row.item(
    0, lambda w: w.as_object(lambda o: o.name("x2"))))).append(lambda v: v.as_indexed(
    lambda row: row.append(lambda w: w.as_object(lambda o: o.name("y")))))).update()
assert p.banks[0][0] is x and x.name == "x2" and p.banks[1][0].name == "y" and p.banks[1][0].owner() is p
# Clearing the list removes every item.
p = B.Part(p).banks(lambda rows: rows.clear()).update()
assert p.banks == ()

## LST-04 · Value objects in a list are copied into another owner and by a clone, and removed with their owner's list

In [ ]:
hub = B.Hub().name("hub").create()
p = B.Part().name("p").ports([lambda o: o.name("a"), lambda o: o.name("b")]).create()
a, b = p.ports
p = B.Part(p).ports(lambda items: items.item(0, lambda v: v.as_object(lambda o: o.wires(lambda x: x.sink(b)).wires(
    lambda x: x.sink(hub))))).update()
before = Plain.ToPlain(store)(Part, p)
# A clone copies every list, and the entries among the copies link the copies.
clone = B.Part(p).clone()
assert clone.ports[0] is not a and clone.ports[1] is not b and clone.ports[0].owner() is clone
assert Plain.ToPlain(store)(Part, clone) == before
assert Plain.ToPlain(store)(Part, p) == before  # the source is untouched
# Setting a value object into another owner's list copies it, with its entries.
q = B.Part().name("q").ports([a]).create()
assert q.ports[0] is not a and q.ports[0].owner() is q and q.ports[0].name == "a"
assert Plain.ToPlain(store)(Part, q)["objects"]["s0"]["ports"] == [
    {"name": "a", "wires": [{"sink": {"$ref": "s1"}}, {"sink": {"$ref": "s2", "$schema": "Hub"}}]}]
# Replacing the list removes the value objects no longer in it, with their entries.
p = B.Part(p).ports([lambda o: o.name("c")]).update()
assert Plain.ToPlain(store)(Hub, hub)["objects"]["s0"]["feeds"] == [{"source": {"$ref": "s1"}}, {"source": {"$ref": "s2"}}]
assert Reachable.of(hub) == [hub, clone, q]  # p's ports are gone, so nothing links p
p = B.Part(clone).clear("ports").update()
assert Reachable.of(hub) == [hub, q]
# A value object removed takes the value objects it holds with it, and their entries.
p = B.Part().racks([lambda r: r.name("r").port(lambda o: o.name("p").wires(lambda x: x.sink(hub)))]).create()
assert p.racks[0].port.owner() is p.racks[0] and Reachable.of(hub) == [hub, q, p]
p = B.Part(p).racks(lambda items: items.remove(0)).update()
assert Reachable.of(hub) == [hub, q]

## LST-05 · Snapshots write a list as an array, its value objects nested with `$id`, and decode it strictly

In [ ]:
hub = B.Hub().name("hub").create()
p = B.Part().name("p").tags(["a", "b"]).grid([[1, 2], []]).shapes([lambda u: u.label("x")]).banks(
    [[], [lambda o: o.name("x"), lambda o: o.name("y")]]).create()
x, y = p.banks[1]
p = B.Part(p).banks(lambda rows: rows.item(1, lambda v: v.as_indexed(lambda row: row.item(
    0, lambda w: w.as_object(lambda o: o.wires(lambda e: e.sink(y)).wires(lambda e: e.sink(hub))))))).update()
text = JSON.ToJSON(store).Reachable(Part, p)
assert text == (
    '{"root": "s0", "objects": {"s0": {"name": "p", "tags": ["a", "b"], "grid": [[1, 2], []], "banks": [[], '
    '[{"$id": "s2", "name": "x", "wires": [{"sink": {"$ref": "s3"}}, {"sink": {"$ref": "s1", "$schema": "Hub"}}]}, '
    '{"$id": "s3", "name": "y", "fed": [{"source": {"$ref": "s2"}}]}]], "shapes": [{"label": "x"}]}, '
    '"s1": {"name": "hub", "feeds": [{"source": {"$ref": "s2"}}]}}}')
back = JSON.FromJSON(B).Reachable(Part, text)
assert JSON.ToJSON(store).Reachable(Part, back) == text
assert back.banks[1][0].owner() is back and back.grid == ((1, 2), ())
assert YAML.ToYAML(store).Reachable(Part, YAML.FromYAML(B).Reachable(Part, YAML.ToYAML(store).Reachable(Part, p))) == \
    YAML.ToYAML(store).Reachable(Part, p)
assert C.OfObject(Part, back).compare(C.OfObject(Part, p)) is None  # its port links another hub, by identity
# Decoding is strict, with paths into the lists.
decode_error(snapshot(tags="a"), "$.objects.s0.tags: a list must be an array, got str")
decode_error(snapshot(tags=["a", 1]), "$.objects.s0.tags[1]: expected str, got int")
decode_error(snapshot(grid=[[1], "x"]), "$.objects.s0.grid[1]: a list must be an array, got str")
decode_error(snapshot(ports=[{"nope": 1}]), "$.objects.s0.ports[0].nope: the value object has no property 'nope'")
decode_error(snapshot(ports=[{"$id": "s1"}, {"$id": "s1"}]), "$.objects.s0.ports[1].$id: 's1' is the symbol of two objects")
decode_error(snapshot(ports=[{"wires": [{"sink": {"$ref": "s9"}}]}]),
             "$.objects.s0.ports[0].wires[0].sink: unresolved reference 's9': no value object in the snapshot has this symbol")
decode_error(snapshot(ports=[{"wires": [{"sink": {"$ref": 1}}]}]),
             "$.objects.s0.ports[0].wires[0].sink: a reference is {'$ref': symbol, '$schema': name}")
decode_error(snapshot(ports=[{"wires": [1]}]), "$.objects.s0.ports[0].wires[0]: an entry must be a mapping, got int")
decode_error(snapshot(shapes=[{}]), "$.objects.s0.shapes[0]: a union value holds exactly one branch, got 0")

## LST-06 · Validation checks each item against the item schema, at its index

In [ ]:
validate = Validators.Validate(B)
p = B.Part().name("p").tags(["a"]).ports([lambda o: o.name("a"), lambda o: o.name("b")]).create()
p = B.Part(p).ports(lambda items: items.item(0, lambda v: v.as_object(lambda o: o.wires(
    lambda x: x.sink(p.ports[1]))))).update()
assert validate(Part, p) == [] and validate.Reachable(Part, p) == []


def written(name, write):
    """A Part whose property `name` holds what `write(OfAny)` writes."""
    return Written(lambda v: v.property(name, lambda q: q.value(write)))


def items(*writes):
    return lambda a: a.as_indexed(lambda items: [items.append(w) for w in writes])


native = lambda value: lambda a: a.as_native(lambda n: n.set(value))
port = lambda value: lambda a: a.as_object(lambda o: o.property("name", lambda q: q.value(native(value))))
assert validate(Part, written("tags", items(native("a"), native(1)))) == ["Part#0.tags[1]: expected str, got int"]
assert validate(Part, written("tags", native("a"))) == ["Part#0.tags: expected a list, got str"]
assert validate(Part, written("name", items())) == ["Part#0.name: expected str, got a list"]
assert validate(Part, written("ports", items(native("x")))) == ["Part#0.ports[0]: expected a value object, got str"]
assert validate(Part, written("ports", items(port("a"), port(2)))) == ["Part#0.ports[1].name: expected str, got int"]
assert validate(Part, written("grid", items(items(native(1)), items(native("2"))))) == \
    ["Part#0.grid[1][0]: expected int, got str"]
# `properties_of` gives a list as an object whose `values` are its items, read the same way.
values = Validators.properties_of(p)
assert values["tags"].values == ["a"] and [Validators.properties_of(o)["name"] for o in values["ports"].values] == ["a", "b"]

## LST-07 · Lists compare item by item: lexicographically for ordered natives, and a prefix is less

In [ ]:
Tags = Part.properties["tags"].type


def tags(*values, schema=Tags):
    recorded = C.OfIndexed(schema)
    for value in values:
        recorded.append(lambda a, value=value: a.as_native(lambda n: n.set(value)))
    return recorded


assert tags("a", "b").compare(tags("a", "b")) == 0 and tags().compare(tags()) == 0
assert tags("a").compare(tags("b")) == -1 and tags("b").compare(tags("a", "z")) == 1
assert tags("a").compare(tags("a", "b")) == -1 and tags("a", "b").compare(tags("a")) == 1
Flags = S.OfIndexed.Builder().of(lambda t: t.as_native(bool)).create()
assert tags(True, schema=Flags).compare(tags(False, schema=Flags)) is None  # booleans are not ordered
assert tags(True, schema=Flags).compare(tags(True, schema=Flags)) == 0
assert tags("a").compare(tags(True, schema=Flags)) is None  # different schemas
# Objects compare their lists deeply, value objects included, and links among them by path.
p = B.Part().tags(["a"]).ports([lambda o: o.name("a"), lambda o: o.name("b")]).create()
p = B.Part(p).ports(lambda items: items.item(0, lambda v: v.as_object(lambda o: o.wires(
    lambda x: x.sink(p.ports[1]))))).update()
clone = B.Part(p).clone()
assert C.OfObject(Part, p).compare(C.OfObject(Part, clone)) == 0
crossed = B.Part(p).clone()
crossed = B.Part(crossed).ports(lambda items: items.item(0, lambda v: v.as_object(lambda o: o.wires(
    lambda x: x.sink(crossed.ports[0]))))).update()  # a self-loop too: not the same links
assert C.OfObject(Part, p).compare(C.OfObject(Part, crossed)) is None
assert C.OfObject(Part, B.Part().tags([]).create()).compare(C.OfObject(Part, B.Part().create())) is None
assert C.OfObject(Part, B.Part().tags(["a"]).create()).compare(C.OfObject(Part, B.Part().tags(["b"]).create())) is None

## LST-08 · Reachability follows the value objects in lists, and entry properties hold lists

In [ ]:
hub = B.Hub().name("hub").create()
p = B.Part().banks([[lambda o: o.name("x")]]).create()
p = B.Part(p).banks(lambda rows: rows.item(0, lambda v: v.as_indexed(lambda row: row.item(
    0, lambda w: w.as_object(lambda o: o.wires(lambda e: e.sink(hub))))))).update()
assert Reachable.of(p) == [p, hub] and Reachable.of(hub) == [hub, p]  # a value object reached brings in its owner
assert Reachable.targets([p]) == {hub.identity()}
# Entries key and compare by their lists, in order: an equal entry is elided.
p = B.Part().labelled(lambda x: x.hub(hub).labels(["a", "b"])).labelled(lambda x: x.hub(hub).labels(("a", "b"))).labelled(
    lambda x: x.hub(hub).labels(["b", "a"])).create()
assert Plain.ToPlain(store)(Part, p)["objects"]["s0"]["labelled"] == [
    {"hub": {"$ref": "s1", "$schema": "Hub"}, "labels": ["a", "b"]},
    {"hub": {"$ref": "s1", "$schema": "Hub"}, "labels": ["b", "a"]}]
back = Plain.FromPlain(B).Reachable(Part, Plain.ToPlain(store).Reachable(Part, p))
assert Plain.ToPlain(store).Reachable(Part, back) == Plain.ToPlain(store).Reachable(Part, p)
assert Validators.Validate(B).Reachable(Part, p) == []

## LST-09 · The list classes behave through their protocols

In [ ]:
seen = []
# Plain's writer: items read back, edited, removed and cleared.
out = {}
writer = Plain._AnyWriter(out, "tags", Part.properties["tags"].type, lambda t: {}, lambda t: None)
writer.as_indexed(lambda items: items.append(lambda a: a.as_native(lambda n: n.set("a"))).append(
    lambda a: a.as_native(lambda n: n.set("b"))))
writer.as_indexed(lambda items: items.items(lambda a: a.as_native(lambda n: seen.append(n.get()))))
writer.as_indexed(lambda items: items.item(1, lambda a: a.as_native(lambda n: n.set("c"))).remove(0))
assert seen == ["a", "b"] and out == {"tags": ["c"]}
writer.as_indexed(lambda items: items.item(0, lambda a: a.as_native(lambda n: n.clear())))  # an item cleared is removed
writer.as_indexed(lambda items: items.append(lambda a: a.as_native(lambda n: n.set("d"))).clear())
assert out == {"tags": []}
with raises(TypeError, match="property 'name' is not a list"):
    Plain._AnyWriter(out, "name", Part.properties["name"].type, lambda t: {}, lambda t: None).as_indexed(lambda items: None)
with raises(LookupError, match="the list has no item 0"):
    writer.as_indexed(lambda items: items.remove(0))
# The validation recorder keeps what it is given, and reads it back.
record = Validators._ObjectRecord()
record.property("tags", lambda p: p.value(lambda a: a.as_indexed(lambda items: items.append(
    lambda v: v.as_native(lambda n: n.set("a"))).append(lambda v: v.as_native(lambda n: n.set("b"))))))
seen.clear()
record.property("tags", lambda p: p.value(lambda a: a.as_indexed(lambda items: items.items(
    lambda v: v.as_native(lambda n: seen.append(n.get()))).item(0, lambda v: v.as_native(lambda n: n.set("z"))).remove(1))))
assert seen == ["a", "b"] and record.values["tags"].values == ["z"]
replayed = Validators._ObjectRecord()
record.accept(replayed)
assert replayed.values["tags"].values == ["z"]
record.property("tags", lambda p: p.value(lambda a: a.as_indexed(lambda items: items.clear())))
assert record.values["tags"].values == []
with raises(LookupError, match="the list has no item 2"):
    record.property("tags", lambda p: p.value(lambda a: a.as_indexed(lambda items: items.remove(2))))
# The comparison recorder too.
recorded = C.OfAny(Part.properties["tags"].type)
recorded.as_indexed(lambda items: items.append(lambda v: v.as_native(lambda n: n.set("a"))).append(
    lambda v: v.as_native(lambda n: n.set("b"))))
seen.clear()
recorded.as_indexed(lambda items: items.items(lambda v: v.as_native(lambda n: seen.append(n.get()))).item(
    0, lambda v: v.as_native(lambda n: n.set("z"))).remove(1))
assert seen == ["a", "b"]
other = C.OfAny(Part.properties["tags"].type)
other.as_indexed(lambda items: items.append(lambda v: v.as_native(lambda n: n.set("z"))).append(
    lambda v: v.as_native(lambda n: n.clear())))  # an item written with no value is left out
assert recorded.compare(other) == 0
recorded.as_indexed(lambda items: items.clear())
assert recorded.compare(other) == -1
named = C.OfAny(Part.properties["name"].type)
named.as_native(lambda n: n.set("z"))
assert recorded.compare(named) is None and named.compare(recorded) is None  # a list and a native are incomparable
with raises(LookupError, match="the list has no item 5"):
    recorded.as_indexed(lambda items: items.remove(5))
with raises(TypeError, match="the schema is not a list schema"):
    C.OfAny(Part.properties["name"].type).as_indexed(lambda items: None)
# The reachability collector discards lists, keeping no items, and follows the value objects appended.
nested = []
collector = Reachable._Collector(seen.append)
collector.property("ports", lambda p: p.value(lambda a: a.as_indexed(nested.append)))
items = nested[0]
assert items.items(seen.append) is items and items.clear() is items
items.append(lambda v: v.as_object(lambda o: nested.append(o)))
assert isinstance(nested[1], Reachable._Collector)
for method in (lambda: items.item(0, seen.append), lambda: items.remove(0)):
    with raises(LookupError, match="the list has no item 0"):
        method()
# The proxy builder: removing an item, and clearing.
p = B.Part().tags(["a", "b", "c"]).create()
p = B.Part(p).tags(lambda items: items.remove(1)).update()
assert p.tags == ("a", "c")

## LST-10 · A key schema makes a list keyed, and an extent bounds a positional one

In [ ]:
Attrs = S.OfIndexed.Builder().key(Str).of(Float).create()
assert not Attrs.positional and Attrs.validate() == [] and Attrs.minimum == 0 and Attrs.capacity is None
Row = S.OfIndexed.Builder().key(Int).of(Str).extent(1, 3).create()  # an int key: positional
assert Row.positional and Row.minimum == 1 and Row.capacity == 3 and Row.validate() == []
assert S.OfIndexed.Builder().of(Str).extent(maximum=4).create().extent == S.OfIndexed.Extent(0, 4)
assert Part.properties["cells"].type.key is Cell and Part.validate() == [] and Labels.validate() == []
# A key is a value: never a reference object, and its value objects have no adjacencies.
Card = S.OfObject.Builder().ref().create()
assert S.OfIndexed.Builder().key(Card).of(Str).create().validate() == ["key: a reference object schema cannot be a key"]
assert S.OfIndexed.Builder().key(Port).of(Str).create().validate() == [
    "key: a value object in a key cannot have adjacencies"]
assert S.OfIndexed.Builder().key(list_of(Port)).of(Str).create().validate() == [
    "key: a value object in a key cannot have adjacencies"]
assert S.OfIndexed.Builder().key(lambda t: t.as_native(lambda n: n.type(int).bits(0))).of(Str).create().validate() == [
    "key: a width in bits must be a positive int, got 0"]
# An extent is for positional lists, of ints, its minimum at most its maximum.
assert S.OfIndexed.Builder().key(Str).of(Str).extent(0, 1).create().validate() == [
    "an extent bounds a positional list, whose keys are ints"]
assert S.OfIndexed.Builder().of(Str).extent(5, 2).create().validate() == ["an extent's minimum 5 exceeds its maximum 2"]
Odd = S.OfIndexed.Data(item=S.OfNative.Data(str), extent=S.OfIndexed.Extent("1", None))
assert Odd.validate() == ["an extent's minimum and maximum are ints, got '1' and None"]
assert Odd.minimum == 0 and Odd.capacity is None  # an extent that is not ints bounds nothing

## LST-11 · Proxies read a keyed list as a read-only mapping, in insertion order, set from a mapping or pairs

In [ ]:
p = B.Part().attrs({"a": 1.5, "b": -0.0}).cells([(lambda c: c.r(1).c(2), 7), (lambda c: c.r(0).c(0), 9)]).create()
assert isinstance(p.attrs, Proxies.OfIndexed.Map) and list(p.attrs.items()) == [("a", 1.5), ("b", -0.0)]
assert p.attrs["a"] == 1.5 and p.attrs.get("z") is None and "b" in p.attrs and len(p.attrs) == 2
assert repr(p.attrs) == "Map([('a', 1.5), ('b', -0.0)])"
with raises(KeyError):
    p.attrs["z"]
with raises(TypeError):
    p.attrs["c"] = 1.0
first = list(p.cells)[0]
assert first.r == 1 and first.owner() is p and p.cells[first] == 7
other = list(B.Part().cells([(lambda c: c.r(1).c(2), 1)]).create().cells)[0]
assert p.cells[other] == 7  # a value object key matches by structure
# `put` replaces a value in place, keeping the order; a new key goes last; the value of a value object keeps its identity.
p = B.Part(p).attrs(lambda m: m.put(lambda k: k.as_native(lambda n: n.set("a")), lambda v: v.as_native(lambda n: n.set(2.0)))
                    .put(lambda k: k.as_native(lambda n: n.set("c")), lambda v: v.as_native(lambda n: n.set(3.0)))).update()
assert list(p.attrs.items()) == [("a", 2.0), ("b", -0.0), ("c", 3.0)]
p = B.Part(p).named({b"x": lambda o: o.name("x")}).update()
x = p.named[b"x"]
p = B.Part(p).named(lambda m: m.put(lambda k: k.as_native(lambda n: n.set(b"x")),
                                    lambda v: v.as_object(lambda o: o.name("x2")))).update()
assert p.named[b"x"] is x and x.name == "x2" and x.owner() is p
assert list(B.Part().attrs(p.attrs).create().attrs.items()) == list(p.attrs.items())  # a Map is a mapping
assert list(B.Part().attrs({}).create().attrs.items()) == []  # empty, unlike absent
# Through the builder: by key and by position.
seen = []
B.Part(p).attrs(lambda m: m.pairs(lambda item: item.key(lambda k: k.as_native(lambda n: seen.append(n.get())))
                                  .value(lambda v: v.as_native(lambda n: seen.append(n.get())))))
assert seen == ["a", 2.0, "b", -0.0, "c", 3.0]
seen.clear()
B.Part(p).attrs(lambda m: m.at(lambda k: k.as_native(lambda n: n.set("b")), lambda v: v.as_native(lambda n: seen.append(n.get())))
                .items(lambda v: v.as_native(lambda n: seen.append(n.get()))))
assert seen == [-0.0, 2.0, -0.0, 3.0]
p = B.Part(p).attrs(lambda m: m.discard(lambda k: k.as_native(lambda n: n.set("b"))).remove(0)).update()
assert list(p.attrs.items()) == [("c", 3.0)]
p = B.Part(p).attrs(lambda m: m.item(0, lambda v: v.as_native(lambda n: n.clear()))  # an item with no value is left out
                    .put(lambda k: k.as_native(lambda n: n.set("d")), lambda v: v.as_native(lambda n: n.clear()))).update()
assert len(p.attrs) == 0
p = B.Part(p).attrs({"a": 1.0}).update()
p = B.Part(p).attrs(lambda m: m.clear()).update()
assert len(p.attrs) == 0
key_a = lambda k: k.as_native(lambda n: n.set("a"))  # noqa: E731
for wrong, error, message in [
    (lambda m: m.append(lambda v: None), TypeError, "a keyed list takes put, not append"),
    (lambda m: m.put(lambda k: None, lambda v: None), ValueError, "a key needs a value"),
    (lambda m: m.at(key_a, lambda v: None), LookupError, "the list has no item with this key"),
    (lambda m: m.discard(key_a), LookupError, "the list has no item with this key"),
]:
    with raises(error, match=message):
        B.Part(p).attrs(wrong)
with raises(TypeError, match="property 'attrs' takes a mapping, (key, value) pairs or a Spec, got str"):
    B.Part().attrs("x")
with raises(TypeError, match="property 'attrs' takes (key, value) pairs, got list"):
    B.Part().attrs([["a"]])
with raises(TypeError, match="property 'tags' takes a list or a Spec, got dict"):
    B.Part().tags({"a": "b"})

## LST-12 · A positional list's keys are its positions, from its extent's minimum

In [ ]:
p = B.Part().row([10, 20]).create()
keys = []
B.Part(p).row(lambda r: r.pairs(lambda item: item.key(lambda k: k.as_native(lambda n: keys.append(n.get())))))
assert keys == [1, 2]
at = lambda key: lambda k: k.as_native(lambda n: n.set(key))  # noqa: E731
seen = []
B.Part(p).row(lambda r: r.at(at(2), lambda v: v.as_native(lambda n: seen.append(n.get()))))
assert seen == [20]
p = B.Part(p).row(lambda r: r.put(at(3), lambda v: v.as_native(lambda n: n.set(30)))   # the next key appends
                  .put(at(1), lambda v: v.as_native(lambda n: n.set(11)))).update()   # a key held replaces
assert p.row == (11, 20, 30)
p = B.Part(p).row(lambda r: r.discard(at(2))).update()
assert p.row == (11, 30)
for wrong, error, message in [
    (lambda r: r.put(at(5), lambda v: None), LookupError, "the list has no item 5"),
    (lambda r: r.at(at(0), lambda v: None), LookupError, "the list has no item 0"),
    (lambda r: r.discard(at(3)), LookupError, "the list has no item 3"),
    (lambda r: r.at(at("1"), lambda v: None), TypeError, "a positional list's keys are ints, got str"),
    (lambda r: r.at(lambda k: None, lambda v: None), ValueError, "a key needs a value"),
]:
    with raises(error, match=message):
        B.Part(p).row(wrong)
# Validation labels a positional list's items by key, and checks its extent.
validate = Validators.Validate(B)
assert validate(Part, B.Part().row([1, 2, 3]).create()) == []
assert validate(Part, B.Part().row([1, 2, 3, 4]).create()) == ["Part#0.row: 4 items are more than the extent 1..3 holds"]
assert validate(Part, Written(lambda v: v.property("row", lambda q: q.value(lambda a: a.as_indexed(
    lambda r: r.append(lambda i: i.as_native(lambda n: n.set(1))).append(lambda i: i.as_native(lambda n: n.set("2"))))))
    )) == ["Part#0.row[2]: expected int, got str"]

## LST-13 · A keyed list is a mapping from its keys' text when that never starts with `$`, else an array of keys and values

In [ ]:
hub = B.Hub().name("hub").create()
p = (B.Part().name("p").attrs({"a": 1.5, "$ref": 2.0})  # str keys may be anything, so they are written as items
     .weights({1.5: "x", float("nan"): "n", -0.0: "z"}).flags({True: "on", False: "off"})
     .named({b"\x00": lambda o: o.name("zero"), b"x": lambda o: o.name("x")})
     .cells([(lambda c: c.r(1).c(2).w(0.5).path([1]).meta({"k": "v"}), 7)]).create())
p = B.Part(p).named(lambda m: m.put(lambda k: k.as_native(lambda n: n.set(b"x")),
                                    lambda v: v.as_object(lambda o: o.wires(lambda e: e.sink(hub))))).update()
text = JSON.ToJSON(store).Reachable(Part, p)
assert text == (
    '{"root": "s0", "objects": {"s0": {"name": "p", "attrs": [{"key": "a", "value": 1.5}, {"key": "$ref", "value": 2.0}], '
    '"weights": {"1.5": "x", "NaN": "n", "-0.0": "z"}, '
    '"flags": {"true": "on", "false": "off"}, "named": {"AA==": {"name": "zero"}, '
    '"eA==": {"$id": "s2", "name": "x", "wires": [{"sink": {"$ref": "s1", "$schema": "Hub"}}]}}, '
    '"cells": [{"key": {"r": 1, "c": 2, "w": 0.5, "path": [1], "meta": [{"key": "k", "value": "v"}]}, "value": 7}]}, '
    '"s1": {"name": "hub", "feeds": [{"source": {"$ref": "s2"}}]}}}')
back = JSON.FromJSON(B).Reachable(Part, text)
assert JSON.ToJSON(store).Reachable(Part, back) == text and back.named[b"x"].owner() is back
assert YAML.ToYAML(store).Reachable(Part, YAML.FromYAML(B).Reachable(Part, YAML.ToYAML(store).Reachable(Part, p))) == \
    YAML.ToYAML(store).Reachable(Part, p)
# Decoding is strict: only a key's canonical text, mappings and arrays where they belong, each key once.
decode_error(snapshot(weights=[]), "$.objects.s0.weights: a keyed list must be a mapping, got list")
decode_error(snapshot(weights={"1.50": "x"}), '$.objects.s0.weights["1.50"]: expected the text of a float key, got \'1.50\'')
decode_error(snapshot(weights={"one": "x"}), "$.objects.s0.weights.one: expected the text of a float key, got 'one'")
decode_error(snapshot(flags={"yes": "x"}), "$.objects.s0.flags.yes: expected the text of a bool key, got 'yes'")
decode_error(snapshot(named={"!!": {}}), '$.objects.s0.named["!!"]: expected the text of a bytes key, got \'!!\'')
decode_error(snapshot(weights={"1.5": True}), '$.objects.s0.weights["1.5"]: expected str, got bool')
decode_error(snapshot(attrs={"a": 1.5}), "$.objects.s0.attrs: a keyed list must be an array, got dict")
decode_error(snapshot(cells={}), "$.objects.s0.cells: a keyed list must be an array, got dict")
decode_error(snapshot(cells=[{"key": {"r": 1}}]),
             "$.objects.s0.cells[0]: an item of a keyed list is {'key': key, 'value': value}")
decode_error(snapshot(cells=[{"key": {"r": "1"}, "value": 1}]), "$.objects.s0.cells[0].key.r: expected int, got str")
decode_error(snapshot(cells=[{"key": {"$id": "s5"}, "value": 1}]),
             "$.objects.s0.cells[0].key.$id: the value object has no property '$id'")
same_key = {"r": 1, "w": 0.5, "path": [1], "meta": [{"key": "k", "value": "v"}]}
decode_error(snapshot(cells=[{"key": same_key, "value": 1}, {"key": same_key, "value": 2}]),
             "$.objects.s0.cells[1].key: the same key as item 0")

## LST-14 · Keyed lists are validated by position, compared as mappings, and followed for reachability

In [ ]:
validate = Validators.Validate(B)
p = B.Part().attrs({"a": 1.0, "b": 2.0}).cells([(lambda c: c.r(1).meta({"k": "v"}), 1)]).create()
assert validate(Part, p) == []


def written(name, *items):
    """A Part whose keyed list `name` holds `items`: (key, value) writers, or a lone value writer appended."""
    def write(lst):
        for item in items:
            if isinstance(item, tuple):
                lst.put(*item)
            else:
                lst.append(item)
    return Written(lambda v: v.property(name, lambda q: q.value(lambda a: a.as_indexed(write))))


native = lambda value: lambda a: a.as_native(lambda n: n.set(value))  # noqa: E731
cell = lambda meta: lambda a: a.as_object(lambda o: o.property("meta", lambda q: q.value(  # noqa: E731
    lambda m: m.as_indexed(lambda lst: lst.put(native("k"), native(meta))))))
assert validate(Part, written("attrs", (native("a"), native(1.0)), (native("a"), native(2.0)))) == [
    "Part#0.attrs[1].key: the same key as item 0"]
assert validate(Part, written("cells", (cell("v"), native(1)), (cell("v"), native(2)), (cell("w"), native(3)))) == [
    "Part#0.cells[1].key: the same key as item 0"]
assert validate(Part, written("attrs", (native(1), native("x")))) == [
    "Part#0.attrs[0].key: expected str, got int", "Part#0.attrs[0]: expected float, got str"]
assert validate(Part, written("attrs", native(1.0))) == ["Part#0.attrs[0].key: not set"]
values = Validators.properties_of(p)["attrs"]
assert values.keys == ["a", "b"] and values.values == [1.0, 2.0]
replayed = Validators._ObjectRecord()
values_record = Validators._ObjectRecord()
p.accept(values_record)
values_record.accept(replayed)  # a recorded keyed list replays its keys
assert replayed.values["attrs"].keys == ["a", "b"]
# Keyed lists compare as mappings: the same keys with equal values, in any order.
compare = lambda a, b: C.OfObject(Part, B.Part().attrs(a).create()).compare(C.OfObject(Part, B.Part().attrs(b).create()))  # noqa: E731
assert compare({"a": 1.0, "b": 2.0}, {"b": 2.0, "a": 1.0}) == 0
assert compare({"a": 1.0}, {"a": 2.0}) is None and compare({"a": 1.0}, {"a": 1.0, "b": 2.0}) is None
assert compare({"a": 1.0}, {"b": 1.0}) is None
# Entries key by keyed lists as mappings too, and reachability follows the value objects they hold.
hub = B.Hub().name("hub").create()
p = B.Part().labelled(lambda x: x.hub(hub).notes({"a": "1", "b": "2"})).labelled(
    lambda x: x.hub(hub).notes({"b": "2", "a": "1"})).named({b"x": lambda o: o.name("x")}).create()
assert len(Plain.ToPlain(store)(Part, p)["objects"]["s0"]["labelled"]) == 1
p = B.Part(p).named(lambda m: m.put(lambda k: k.as_native(lambda n: n.set(b"x")),
                                    lambda v: v.as_object(lambda o: o.wires(lambda e: e.sink(hub))))).update()
assert Reachable.of(p) == [p, hub] and Reachable.of(hub) == [hub, p]
p = B.Part(p).named({}).update()  # the value object is removed, and its entries
assert "feeds" not in Plain.ToPlain(store)(Hub, hub)["objects"]["s0"]

## LST-15 · The keyed list classes behave through their protocols

In [ ]:
key = lambda value: lambda a: a.as_native(lambda n: n.set(value))  # noqa: E731
seen = []
read = lambda a: a.as_native(lambda n: seen.append(n.get()))  # noqa: E731
item_keys = lambda lst: lst.pairs(lambda i: i.key(read).value(read))  # noqa: E731
# Plain's keyed writers: an array of keys and values (str keys), and a mapping by the keys' text (float keys).
for name, k1, k2, expected, empty in [("attrs", "a", "b", {"attrs": [{"key": "b", "value": 2.5}]}, {"attrs": []}),
                                     ("weights", 1.5, 2.5, {"weights": {"2.5": "y"}}, {"weights": {}})]:
    out = {}
    schema = Part.properties[name].type
    value = (lambda v: lambda a: a.as_native(lambda n: n.set(v)))
    v1, v2 = (1.5, 2.5) if name == "attrs" else ("x", "y")
    writer = Plain._AnyWriter(out, name, schema, lambda t: {}, lambda t: None)
    writer.as_indexed(lambda lst: lst.put(key(k1), value(v1)).put(key(k2), value(v1)).put(key(k2), value(v2)))
    seen.clear()
    writer.as_indexed(lambda lst: item_keys(lst).at(key(k2), read).items(read))  # a writer reopened reads its own keys
    assert seen == [k1, v1, k2, v2, v2, v1, v2]
    writer.as_indexed(lambda lst: lst.discard(key(k1)))
    assert out == expected
    writer.as_indexed(lambda lst: lst.item(0, lambda a: a.as_native(lambda n: n.clear())))
    writer.as_indexed(lambda lst: lst.put(key(k1), value(v1)).remove(0).clear())
    assert out == empty
    for wrong, error, message in [(lambda lst: lst.append(read), TypeError, "a keyed list takes put, not append"),
                                  (lambda lst: lst.at(key(k1), read), LookupError, "the list has no item with this key"),
                                  (lambda lst: lst.put(lambda a: None, read), ValueError, "a key needs a value")]:
        with raises(error, match=message):
            writer.as_indexed(wrong)
out = {}
writer = Plain._AnyWriter(out, "cells", Part.properties["cells"].type, lambda t: {}, lambda t: None)
cell_key = lambda r: lambda a: a.as_object(lambda o: o.property("r", lambda q: q.value(key(r))))  # noqa: E731
writer.as_indexed(lambda lst: lst.put(cell_key(1), key(10)).put(cell_key(2), key(20)).put(cell_key(1), key(11)))
assert out == {"cells": [{"key": {"r": 1}, "value": 11}, {"key": {"r": 2}, "value": 20}]}
seen.clear()
writer.as_indexed(lambda lst: lst.pairs(lambda i: i.value(read)))
assert seen == [11, 20]
# Plain's positional writer, by key.
out = {}
writer = Plain._AnyWriter(out, "row", Part.properties["row"].type, lambda t: {}, lambda t: None)
writer.as_indexed(lambda lst: lst.put(key(1), key(10)).put(key(2), key(20)).put(key(1), key(11)))
seen.clear()
writer.as_indexed(lambda lst: item_keys(lst).at(key(2), read).discard(key(1)))
assert seen == [1, 11, 2, 20, 20] and out == {"row": [20]}
for wrong, error, message in [(lambda lst: lst.at(key(7), read), LookupError, "the list has no item 7"),
                              (lambda lst: lst.at(key("1"), read), TypeError, "expected int, got str"),
                              (lambda lst: lst.at(lambda a: None, read), ValueError, "a key needs a value")]:
    with raises(error, match=message):
        writer.as_indexed(wrong)
# The validation recorder records keys, finds items by them, and keeps every item put.
record = Validators._ListRecord()
record.put(key("a"), key(1.0)).put(key("b"), key(2.0)).put(key("a"), key(3.0)).append(key(4.0))
seen.clear()
item_keys(record).at(key("b"), read)
assert seen == ["a", 1.0, "b", 2.0, "a", 3.0, None, 4.0, 2.0]
record.discard(key("b"))
assert record.keys == ["a", "a", None] and record.values == [1.0, 3.0, 4.0]
with raises(LookupError, match="the list has no item with this key"):
    record.at(key("z"), read)
seen.clear()
record.at(lambda a: None, read)  # no key written: an item appended, which has none
assert seen == [4.0]
# The comparison recorder: a keyed list by key, a positional one by position from its minimum.
keyed_list = C.OfIndexed(Part.properties["attrs"].type)
keyed_list.put(key("a"), key(1.0)).put(key("b"), key(2.0)).put(key("a"), key(3.0))
seen.clear()
item_keys(keyed_list).at(key("b"), read)
assert seen == ["a", 3.0, "b", 2.0, 2.0]
keyed_list.discard(key("b")).put(key("c"), lambda a: a.as_native(lambda n: n.clear()))  # no value: left out
assert len(keyed_list._items) == 1
for wrong, error, message in [(lambda: keyed_list.append(read), TypeError, "a keyed list takes put, not append"),
                              (lambda: keyed_list.at(key("z"), read), LookupError, "the list has no item with this key"),
                              (lambda: keyed_list.put(lambda a: None, read), ValueError, "a key needs a value")]:
    with raises(error, match=message):
        wrong()
row = C.OfIndexed(Part.properties["row"].type)
row.put(key(1), key(10)).put(key(2), key(20)).put(key(1), key(11))
seen.clear()
item_keys(row).at(key(2), read)
assert seen == [1, 11, 2, 20, 20]
row.discard(key(1))
with raises(LookupError, match="the list has no item 3"):
    row.at(key(3), read)
# The reachability collector keeps no items, but follows what is put.
nested = []
collector = Reachable._Collector(seen.append)
collector.property("named", lambda p: p.value(lambda a: a.as_indexed(nested.append)))
discarded = nested[0]
assert discarded.pairs(seen.append) is discarded
discarded.put(lambda k: nested.append(k), lambda v: v.as_object(lambda o: nested.append(o)))
assert isinstance(nested[2], Reachable._Collector)
for method in (lambda: discarded.at(key(b"x"), read), lambda: discarded.discard(key(b"x"))):
    with raises(LookupError, match="the list has no item with this key"):
        method()